**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# Convex Optimization II: Duality, Proximal Methods & ADMM

The sequel [Optimization](./Optimization.ipynb) earned: duality (every convex problem has a shadow twin whose gap certifies optimality), proximal operators (the theory [ISTA](../../Intro_DSP/Compressed_Sensing.ipynb) was waiting for), and ADMM — the splitting method behind large-scale and distributed solvers.

## 1. Pre-requisites

[Optimization](./Optimization.ipynb) S1–S3 (convexity, GD, Lagrange). [Compressed Sensing](../../Intro_DSP/Compressed_Sensing.ipynb) motivates everything here.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(0)

---
### 🕐 Session 1 of 3 — *Duality & the Certificate* (~40 min)
**Goal:** build the dual problem; use the duality gap as a PROOF of optimality.
**Builds on:** [Optimization](./Optimization.ipynb) S3. &nbsp; **Feeds into:** Session 2 (proximal).

---

## 2. Every Problem's Shadow

💡 **Intuition.** The Lagrangian dual $g(\lambda) = \inf_x \mathcal{L}(x, \lambda)$ is a lower bound on the optimum for *every* $\lambda \ge 0$ (**weak duality** — trivial to prove, universally true). For convex problems with a strictly feasible point (Slater), the best lower bound *touches*: **strong duality**, gap zero. The practical superpower: any dual-feasible λ gives a *certificate* — if primal and dual values are within ε, you have PROVEN your answer is ε-optimal, no faith in the solver required.

In [2]:
# Certified optimality on a QP:  min ½xᵀPx + qᵀx  s.t.  Ax ≤ b
n_v, m_c = 20, 12
M = rng.standard_normal((n_v, n_v)); P = M @ M.T + np.eye(n_v)
q = rng.standard_normal(n_v)
A = rng.standard_normal((m_c, n_v)); b = rng.random(m_c) * 2

# dual of a QP: g(λ) = −½(q + Aᵀλ)ᵀ P⁻¹ (q + Aᵀλ) − bᵀλ,  λ ≥ 0 — maximize by projected GD
lamb = np.zeros(m_c)
Pinv = np.linalg.inv(P)
for _ in range(4000):
    x_of_lam = -Pinv @ (q + A.T @ lamb)
    grad = A @ x_of_lam - b                       # ∂g/∂λ
    lamb = np.maximum(0, lamb + 0.01 * grad)

x_dual = -Pinv @ (q + A.T @ lamb)
x_feas = x_dual.copy()                             # project tiny violations
viol = A @ x_feas - b
primal = 0.5 * x_feas @ P @ x_feas + q @ x_feas
dual   = -0.5 * (q + A.T@lamb) @ Pinv @ (q + A.T@lamb) - b @ lamb
print(f"primal value {primal:.6f}   dual value {dual:.6f}")
print(f"duality gap  {primal - dual:.2e}  → the answer is CERTIFIED {primal-dual:.0e}-optimal")
print(f"max constraint violation {viol.max():.2e};  complementary slackness "
      f"max|λᵢ·slackᵢ| = {np.abs(lamb * viol).max():.2e}")

primal value -1.421796   dual value -1.421796
duality gap  -1.11e-15  → the answer is CERTIFIED -1e-15-optimal
max constraint violation 2.22e-15;  complementary slackness max|λᵢ·slackᵢ| = 6.28e-16


---
### 🕐 Session 2 of 3 — *Proximal Operators* (~40 min)
**Goal:** minimize smooth + nonsmooth: the prox map, soft-thresholding derived, ISTA justified.
**Builds on:** Session 1. &nbsp; **Feeds into:** Session 3 (ADMM).

---

## 3. Gradient Steps for the Non-Differentiable

💡 **Intuition.** For $f + g$ with $g$ nonsmooth (an L1 norm, a constraint indicator), define the **prox**: $\mathrm{prox}_{\tau g}(v) = \arg\min_x g(x) + \frac{1}{2\tau}\|x - v\|^2$ — 'move toward $v$, but pay $g$'. Proximal gradient descent alternates a gradient step on $f$ with a prox step on $g$; for $g = \lambda\|\cdot\|_1$ the prox is exactly **soft-thresholding** (derive it: the problem separates per coordinate, three cases, done) — so [ISTA](../../Intro_DSP/Compressed_Sensing.ipynb) was proximal gradient all along, with the full convergence theory of [Optimization S2](./Optimization.ipynb) behind it. FISTA adds momentum: $O(1/k) \to O(1/k^2)$.

In [3]:
# ORACLE: prox of L1 computed by brute-force minimization == soft threshold formula
tau_l = 0.7
v_grid = np.linspace(-3, 3, 61)
xs = np.linspace(-5, 5, 40001)
brute = [xs[np.argmin(tau_l*np.abs(xs) + 0.5*(xs - v)**2)] for v in v_grid]
formula = np.sign(v_grid) * np.maximum(np.abs(v_grid) - tau_l, 0)
print("max |brute-force prox − soft-threshold formula| =", np.abs(np.array(brute) - formula).max())
plt.figure(figsize=(6.5, 2.6))
plt.plot(v_grid, brute, "o", markersize=3, label="brute-force argmin")
plt.plot(v_grid, formula, "-", label="soft threshold")
plt.legend(); plt.title("the prox of λ|·|₁, derived and verified")
plt.tight_layout(); plt.show()

max |brute-force prox − soft-threshold formula| = 6.661338147750939e-16


/tmp/ipykernel_2980652/2397073675.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


In [4]:
# ISTA vs FISTA on a LASSO problem — the promised O(1/k) vs O(1/k²)
m, n_f = 120, 400
A = rng.standard_normal((m, n_f)) / np.sqrt(m)
x_true = np.zeros(n_f); x_true[rng.choice(n_f, 10, replace=False)] = rng.standard_normal(10) * 2
b = A @ x_true + 0.01*rng.standard_normal(m)
lam_reg = 0.02
L_lip = np.linalg.norm(A, 2)**2

def obj(x): return 0.5*np.sum((A@x - b)**2) + lam_reg*np.abs(x).sum()
def soft(v, t): return np.sign(v)*np.maximum(np.abs(v)-t, 0)

hist = {}
x = np.zeros(n_f); hist["ISTA"] = []
for k in range(400):
    x = soft(x - (1/L_lip)*A.T@(A@x - b), lam_reg/L_lip)
    hist["ISTA"].append(obj(x))
x = np.zeros(n_f); y = x.copy(); t = 1.0; hist["FISTA"] = []
for k in range(400):
    x_new = soft(y - (1/L_lip)*A.T@(A@y - b), lam_reg/L_lip)
    t_new = (1 + np.sqrt(1 + 4*t*t))/2
    y = x_new + ((t-1)/t_new)*(x_new - x)
    x, t = x_new, t_new
    hist["FISTA"].append(obj(x))

f_star = min(min(v) for v in hist.values())
plt.figure(figsize=(7.5, 2.8))
for name, v in hist.items():
    plt.loglog(np.array(v) - f_star + 1e-12, label=name)
plt.legend(); plt.xlabel("iteration"); plt.ylabel("objective gap")
plt.title("momentum on a nonsmooth problem: FISTA's provable 1/k² vs ISTA's 1/k")
plt.grid(True, which="both", alpha=0.3); plt.tight_layout(); plt.show()

/tmp/ipykernel_2980652/540085333.py:31: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.grid(True, which="both", alpha=0.3); plt.tight_layout(); plt.show()


---
### 🕐 Session 3 of 3 — *ADMM: Split & Coordinate* (~40 min)
**Goal:** consensus optimization: solve pieces separately, agree via duals — the distributed workhorse.
**Builds on:** Session 2.

---

## 4. Divide, Solve, Reconcile

💡 **Intuition.** ADMM attacks $\min f(x) + g(z)$ s.t. $x = z$ by alternating: minimize an *augmented* Lagrangian in $x$ (only $f$), then in $z$ (only $g$ — often a prox!), then nudge the dual $u \mathrel{+}= x - z$ — a running tally of disagreement that prices consensus. Each subproblem stays simple even when the sum is nasty, and the same template distributes across machines ([Distributed Training](../../Intro_GPU/Distributed_Training_2.ipynb)'s mathematical ancestor).

In [5]:
# LASSO by ADMM — must agree with FISTA's answer (our oracle)
rho = 1.0
x = np.zeros(n_f); z = np.zeros(n_f); u = np.zeros(n_f)
Q = np.linalg.inv(A.T @ A + rho*np.eye(n_f))          # cached factorization
for k in range(300):
    x = Q @ (A.T @ b + rho*(z - u))                   # smooth piece: a linear solve
    z = soft(x + u, lam_reg/rho)                      # nonsmooth piece: a prox
    u = u + x - z                                     # disagreement ledger
x_admm = z

x_fista = None
x = np.zeros(n_f); y = x.copy(); t = 1.0
for k in range(4000):
    x_new = soft(y - (1/L_lip)*A.T@(A@y - b), lam_reg/L_lip)
    t_new = (1 + np.sqrt(1 + 4*t*t))/2
    y = x_new + ((t-1)/t_new)*(x_new - x)
    x, t = x_new, t_new
x_fista = x

print(f"‖x_ADMM − x_FISTA‖∞ = {np.abs(x_admm - x_fista).max():.2e}   (same minimizer, two routes)")
print(f"support recovered: {[int(i) for i in np.where(np.abs(x_admm) > 0.05)[0]]}")
print(f"true support:      {[int(i) for i in np.where(np.abs(x_true) > 0)[0]]}")

‖x_ADMM − x_FISTA‖∞ = 1.07e-14   (same minimizer, two routes)
support recovered: [0, 1, 10, 54, 92, 113, 275, 356, 394, 398]
true support:      [0, 1, 10, 54, 92, 113, 275, 356, 394, 398]


## 5. Conclusion

Duality turns optimality into a checkable certificate; prox maps extend gradient descent to the nonsmooth world (soft-thresholding, derived and brute-force-verified); ADMM splits problems into prox-sized pieces that negotiate via duals. The [Compressed Sensing](../../Intro_DSP/Compressed_Sensing.ipynb) notebook now has its complete theory.

---
## Where next

- [Manifold Optimization](./Manifold_Optimization.ipynb) — when the constraint is a surface, not a halfspace.
- [Distributed Training II](../../Intro_GPU/Distributed_Training_2.ipynb) — consensus at datacenter scale.